<a href="https://colab.research.google.com/github/poevy/id-speech-transcript/blob/main/UKM_Live_Notulen_ID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UKM Live Notulen — ASR Bahasa Indonesia yang Tahan Noise (v2)
**COMP6822001 Speech Recognition — Final Project**

Notebook ini dipakai untuk **training, evaluasi, dan export** model. Aplikasi web-nya (Next.js di Vercel) dikerjakan sendiri oleh tim dan tidak jalan di Colab — model hasil notebook ini di-export ke ONNX lalu dijalankan **di browser user** (Transformers.js, WebGPU dengan fallback WASM), jadi inference tetap lokal di device user dan tidak memakai external ASR API.

Perubahan v2 (keputusan final): (1) **hanya Bahasa Indonesia** — tanpa code-switching dan tanpa data Inggris, (2) **tanpa fitur terjemahan**. Akibatnya ketahanan noise naik status dari eksperimen samping menjadi **kebutuhan inti**: meeting UKM sering terjadi di ruangan ramai.

**Definisi "tahan noise" yang diukur** (bukan klaim):
- **(a)** WER model fine-tuned <= WER model pretrained yang sama pada SETIAP kondisi noise, diputuskan dengan bootstrap confidence interval.
- **(b)** whisper-base yang dilatih dengan augmentasi noise mengalahkan whisper-base yang dilatih clean-only pada kondisi noise; berapa pun "biaya" WER-nya di kondisi bersih, dilaporkan.
- **(c)** Kurva WER-vs-SNR per jenis noise pada noise yang **tidak pernah dilihat training**, plus contoh hipotesis per kondisi noise.

| Bagian | Isi | Tahap |
|---|---|---|
| 1 | Setup & config | 1 |
| 2 | Data: FLEURS + INDspeech (News, Teldialog), pembagian speaker, test set | 1 |
| 3 | **MFCC manual dari nol** + VAD dari c0 | 1 |
| 4 | Bank noise (MUSAN, DEMAND, babble, overlap) + `add_noise` + set uji noise | 1 |
| 5 | **Eksperimen 13 vs 12 koefisien** (speaker ID + verifikasi, keras) | 1 |
| 6 | Baseline pretrained + tuning argumen decode (hanya di validation) | 1 |
| 7 | Fine-tune whisper-base + **ablation clean-only** | 1 |
| 8 | Evaluasi gabungan: tabel WER + CI, kurva WER-vs-SNR, contoh hipotesis, error analysis | 1 |
| 9 | Simulasi streaming (partial, latency, RTF) | menyusul |
| 10 | Label speaker (siapa bilang apa) | menyusul |
| 11 | Export ONNX q8 + upload Hugging Face Hub | menyusul |

> **Runtime -> Change runtime type -> T4 GPU** sebelum mulai.
> Urutan Bagian 3-4 sengaja ditukar dari spesifikasi (MFCC dulu, noise kemudian) karena `add_noise` butuh VAD dari c0 hasil MFCC manual.
> Tahap 1 = Bagian 1-8 (whisper-base end-to-end dulu, sesuai rencana bertahap); Bagian 9-11 ditambahkan setelah Bagian 1-8 confirmed jalan di Colab.

## 1. Setup

In [ ]:
# Versi di-pin supaya training + export ONNX konsisten (resep v1 yang sudah terbukti di Colab).
!pip -q install "transformers==4.46.3" "datasets==3.1.0" "optimum[onnxruntime]==1.23.3" \
    "accelerate==1.1.1" jiwer librosa soundfile scikit-learn
# Colab memasang 'diffusers' versi baru yang bentrok dengan versi pin saat export ONNX -> hapus.
!pip -q uninstall -y diffusers
# Kalau cell import di bawah gagal setelah install -> Runtime > Restart session, lanjut dari cell berikutnya.

In [ ]:
import os, sys, json, time, shutil, functools, gc, random, zipfile, zlib, io, glob
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import torch, scipy.fftpack
import librosa, jiwer, soundfile as sf
from datasets import load_dataset, Audio, Dataset, concatenate_datasets

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ====================== CONFIG ======================
SR          = 16000
N_MFCC      = 13          # c0..c12
LANGUAGE    = "indonesian"
SEED        = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- training ---
MAX_STEPS   = 1500        # ditentukan sekali dari kurva val-WER whisper-base (mulai 1500, resep v1)
P_NOISE     = 0.6         # probabilitas augmentasi noise per sampel training (ablation: 0)
SNR_LO, SNR_HI = 0.0, 20.0   # SNR augmentasi uniform saat training
SNR_TEST    = [20, 10, 5]     # SNR kondisi uji (INCLUDE_SNR0=True menambah 0 dB)
INCLUDE_SNR0 = False
NOISE_KINDS = ["babble", "overlap", "env"]
LONG_TURN_P = 0.3         # porsi data Indspeech yang digabung jadi turn panjang 10-25 s (menjembatani gap gaya bicara)

# --- pembagian speaker (berdasarkan NOMOR speaker Ind001..Ind400, per korpus) ---
N_VAL_SPK, N_TEST_SPK, N_INTERFERER_SPK = 4, 10, 10

# --- data & hasil ---
FLEURS_TEST_CAP = 200
FLEURS_VAL_N    = 180
IND_VAL_PER_SPK = 30
DRIVE_ROOT  = "/content/drive/MyDrive/notulen_id_v2"
CACHE_DATA_TO_DRIVE = True   # cache data hasil download di Drive supaya reset runtime murah
BS_EVAL     = 16            # batch inference evaluasi
# ====================================================

In [ ]:
try:
    from google.colab import drive; drive.mount("/content/drive")
except Exception as e:
    print("Bukan di Colab / mount dilewati:", e)
for d in ["data", "ckpt", "eval", "logs"]:
    os.makedirs(os.path.join(DRIVE_ROOT, d), exist_ok=True)
try:
    if os.path.isdir("/content/drive/MyDrive"):
        st = shutil.disk_usage("/content/drive/MyDrive")
        print(f"Drive free: {st.free/1e9:.1f} GB (dipakai utk cache data ~8.5 GB + model + hasil eval)")
except Exception as e:
    print("Cek kuota Drive dilewati:", e)

## 2. Data

| Sumber | Ukuran | Gaya | Lisensi | Akses |
|---|---|---|---|---|
| FLEURS `id_id` train/val/test | 2.579 / 350 / 687 clip (~9.1 / 1.2 / 2.4 jam) | kalimat terbaca | CC-BY 4.0 | Hugging Face `google/fleurs` |
| INDspeech_NEWS_LVCSR | 44.000 clip, ~43 jam, 400 speaker | kalimat berita dibaca | **CC BY-NC-SA 4.0** | GitHub `s-sakti/data_indsp_news_lvcsr` |
| INDspeech_TELDIALOG_LVCSR | 40.000 clip, ~36 jam, 400 speaker | kalimat domain telepon dibaca | **CC BY-NC-SA 4.0** | GitHub `s-sakti/data_indsp_teldialog_lvcsr` |

Total ~88 jam dari ~800 speaker. **Tidak ada data percakapan spontan berlisensi layak** untuk bahasa Indonesia — gap gaya antara kalimat dibaca dan meeting asli dicatat sebagai limitation di laporan; yang bisa diserang dengan augmentasi noise adalah robustness akustik.

**Aturan lisensi Indspeech (WAJIB dipegang):** non-komersial, share-alike, **dilarang re-hosting**. Audio TIDAK PERNAH di-upload ke Hub/Drive publik — cache Drive di atas bersifat privat. Lisensi WAJIB disebut di model card dan laporan; cek kebijakan data mata kuliah sebelum mempublikasi weight yang dilatih dengannya.

**Format file Indspeech** (dicek langsung dari repo): zip per speaker `speech/Ind{k}/Ind{xxx}.zip` (`k = (xxx-1)//100`), isi `Ind{xxx}_{G}_{A}_C_{corpus}_{uuuu}.wav` — mis. `Ind001_F_B_C_news_0000.wav` (16 kHz, mono, 16-bit). Transkrip satu file per utterance di `text/all_transcript.zip`: format `|S|` lalu **satu kata per baris** (huruf kecil, tanpa tanda baca) lalu `|E|`. Nomor speaker IndXXX konsisten antara korpus News dan Teldialog (gender+aksen sama untuk sampel yang dicek) -> **pembagian peran memakai NOMOR speaker yang sama untuk kedua korpus**, jadi satu orang tidak mungkin muncul di dua peran berbeda.

**Peran speaker** (seluruh speaker, bukan per clip — tidak ada clip uji yang pernah dilihat training):
| Peran | Sumber |
|---|---|
| Training | semua FLEURS train + speaker Indspeech sisanya (376 nomor) |
| Validation bersih (pilih checkpoint terbaik) | FLEURS validation (subsample) + 4 speaker/corpus Indspeech (maks 30 clip/speaker) |
| Test | FLEURS test (cap 200) + SEMUA clip dari 10 speaker uji/corpus (`indspeech_holdout`) |
| Interferer noise TRAINING | 300 clip dari speaker training |
| Interferer noise TEST | 300 clip dari 10 speaker interferer lain/corpus (beda dari speaker uji) |

**Higiene v2 (memperbaiki cacat v1):** setiap dataset diberi kolom `is_eval`. Collator membaca `is_eval` dan hanya memberi augmentasi noise jika False — jadi validation saat training selalu bersih dan bebas kebocoran test (v1 memakai 100 clip pertama dari test sebagai validation + ikut di-augmentasi).

In [ ]:
def load_fleurs(lang, split):
    # Parquet branch dulu (tanpa loading script), fallback ke repo default (resep v1).
    try:
        ds = load_dataset("parquet", split=split, data_files={split:
              f"hf://datasets/google/fleurs@refs%2Fconvert%2Fparquet/{lang}/{split}/*.parquet"})
    except Exception as e:
        print("parquet gagal, fallback:", repr(e)[:200])
        ds = load_dataset("google/fleurs", lang, split=split, trust_remote_code=True)
    ds = ds.cast_column("audio", Audio(sampling_rate=SR))
    ds = ds.rename_column("raw_transcription", "text")
    return ds.select_columns(["audio", "text"])

fleurs_train = load_fleurs("id_id", "train")            # ~2 GB, sabar
fleurs_val   = load_fleurs("id_id", "validation")
fleurs_test  = load_fleurs("id_id", "test")
print(fleurs_train, fleurs_val, fleurs_test, sep="\n")
print("Contoh teks:", fleurs_train[0]["text"])

In [ ]:
# ==== Download INDspeech (News + Teldialog): zip per speaker dari GitHub ====
# 400 zip/korpus x ~8.6 MB (News ~3.4 GB, Teldialog ~2.9 GB), paralel 8 thread; lalu cache ke Drive.
INDSP = {
    "news":      "s-sakti/data_indsp_news_lvcsr",
    "teldialog": "s-sakti/data_indsp_teldialog_lvcsr",
}
IND_DIR = "/content/indsp"

def _fetch(url, path, tries=3):
    for _ in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=120) as r, open(path, "wb") as f:
                shutil.copyfileobj(r, f)
            if os.path.getsize(path) > 100_000:
                return True
        except Exception:
            time.sleep(2)
    print("GAGAL unduh:", url)
    return False

def ensure_indspeech():
    jobs = []
    for corp, repo in INDSP.items():
        base = f"{IND_DIR}/{corp}"
        if os.path.isdir(f"{base}/wav") and sum(len(fs) for _, _, fs in os.walk(f"{base}/wav")) >= 40_000:
            print(f"{corp}: sudah ada di lokal, skip"); continue
        drive_copy = f"{DRIVE_ROOT}/data/indsp/{corp}"
        if CACHE_DATA_TO_DRIVE and os.path.isdir(f"{drive_copy}/wav") and \
                sum(len(fs) for _, _, fs in os.walk(f"{drive_copy}/wav")) >= 40_000:
            print(f"{corp}: restore dari cache Drive (2-5 menit)...")
            shutil.copytree(drive_copy, base, dirs_exist_ok=True); continue
        os.makedirs(f"{base}/zips", exist_ok=True)
        for spk in range(1, 401):
            zp = f"{base}/zips/Ind{spk:03d}.zip"
            if not os.path.exists(zp):
                url = f"https://raw.githubusercontent.com/{repo}/main/speech/Ind{(spk-1)//100}/Ind{spk:03d}.zip"
                jobs.append((url, zp))
        os.makedirs(f"{base}/textzip", exist_ok=True)
        jobs.append((f"https://raw.githubusercontent.com/{repo}/main/text/all_transcript.zip",
                     f"{base}/textzip/all_transcript.zip"))
    if not jobs:
        return
    print(f"Mengunduh {len(jobs)} file dari GitHub (5-10 menit)...")
    t0, ok = time.time(), 0
    with ThreadPoolExecutor(8) as ex:
        futs = [ex.submit(_fetch, u, p) for u, p in jobs]
        for i, f in enumerate(as_completed(futs)):
            ok += f.result()
            if (i + 1) % 50 == 0:
                print(f"  {i+1}/{len(jobs)} file ({time.time()-t0:.0f}s)")
    assert ok == len(jobs), "Ada file yang gagal diunduh - jalankan cell ini lagi (resume otomatis)."
    # Ekstrak: wav per speaker + transkrip
    for corp in INDSP:
        base = f"{IND_DIR}/{corp}"
        os.makedirs(f"{base}/wav", exist_ok=True)
        nz = 0
        for spk in range(1, 401):
            zp = f"{base}/zips/Ind{spk:03d}.zip"
            with zipfile.ZipFile(zp) as z:
                z.extractall(f"{base}/wav")
            nz += 1
        with zipfile.ZipFile(f"{base}/textzip/all_transcript.zip") as z:
            z.extractall(f"{base}/text")
        shutil.rmtree(f"{base}/zips"); shutil.rmtree(f"{base}/textzip")
        print(f"{corp}: {nz} zip diekstrak -> {base}/wav, transkrip -> {base}/text")
        if CACHE_DATA_TO_DRIVE and os.path.isdir("/content/drive/MyDrive"):
            dst = f"{DRIVE_ROOT}/data/indsp/{corp}"
            print(f"{corp}: copy ke cache Drive (sekali saja, 2-5 menit)...")
            shutil.copytree(base, dst, dirs_exist_ok=True,
                            ignore=shutil.ignore_patterns("*.zip"))

ensure_indspeech()
wav_news  = sum(len(fs) for _, _, fs in os.walk(f"{IND_DIR}/news/wav"))
wav_tel   = sum(len(fs) for _, _, fs in os.walk(f"{IND_DIR}/teldialog/wav"))
print(f"News: {wav_news} wav | Teldialog: {wav_tel} wav (target: 44000 / 40000)")

In [ ]:
# ==== Parsing transkrip + pembagian peran speaker ====
def read_transcript(path):
    # Format file transkrip: |S| lalu satu kata per baris lalu |E| (huruf kecil, tanpa tanda baca).
    words = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line in ("|S|", "|E|"):
                continue
            words.append(line)
    return " ".join(words)

def parse_wav_name(fname):
    # Ind001_F_B_C_news_0000.wav -> (nomor speaker=1, utt="news_0000")
    # utt = 2 bagian terakhir (korpus + nomor) karena file transkrip = {utt}.txt
    stem = fname[:-4]
    parts = stem.split("_")
    return int(parts[0][3:]), "_".join(parts[-2:])

def assign_speaker_roles(n_spk=400, seed=SEED, n_val=N_VAL_SPK, n_test=N_TEST_SPK, n_inter=N_INTERFERER_SPK):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n_spk) + 1
    return {"val": set(perm[:n_val].tolist()),
            "test": set(perm[n_val:n_val+n_test].tolist()),
            "interferer": set(perm[n_val+n_test:n_val+n_test+n_inter].tolist())}

# Bangun records: path, text, speaker (nomor), corpus, utt_id
ROLES = assign_speaker_roles()
ALL_REC = {"news": [], "teldialog": []}
missing_text = 0
for corp in ALL_REC:
    text_dir = f"{IND_DIR}/{corp}/text/all_transcript"
    for spk in range(1, 401):
        spk_dir = f"{IND_DIR}/{corp}/wav/Ind{spk:03d}"
        if not os.path.isdir(spk_dir):
            continue
        for wav in sorted(os.listdir(spk_dir)):
            if not wav.endswith(".wav"):
                continue
            num, utt = parse_wav_name(wav)
            tp = os.path.join(text_dir, f"{utt}.txt")
            if not os.path.exists(tp):
                missing_text += 1; continue
            text = read_transcript(tp)
            if not text.strip():
                continue
            ALL_REC[corp].append(dict(path=os.path.join(spk_dir, wav), text=text,
                                       speaker=num, corpus=corp, utt=utt))
print("Jumlah clip per korpus:", {k: len(v) for k, v in ALL_REC.items()},
      "| transkrip hilang:", missing_text)
print("Peran speaker (nomor 1-400):",
      {r: sorted(list(s))[:12] for r, s in ROLES.items()})

In [ ]:
# ==== Durasi + split peran menjadi records train/val/test/interferer ====
def duration_of(path):
    try:
        return sf.info(path).frames / sf.info(path).samplerate
    except Exception:
        return 0.0

TRAIN_REC, VAL_REC, TEST_REC, INTERFERER_REC = {}, {"news": [], "teldialog": []}, {"news": [], "teldialog": []}, {"news": [], "teldialog": []}
rng_v = np.random.default_rng(SEED + 1)
for corp, recs in ALL_REC.items():
    by_role = {"train": [], "val": [], "test": [], "interferer": []}
    for r in recs:
        role = next((rl for rl, s in ROLES.items() if r["speaker"] in s), "train")
        r["dur"] = duration_of(r["path"])
        by_role[role].append(r)
    # train: buang clip > 30 s (jendela Whisper) — clip Indspeech normalnya 2-8 s
    by_role["train"] = [r for r in by_role["train"] if r["dur"] <= 30.0]
    # val: maks IND_VAL_PER_SPK clip per speaker, deterministik
    by_spk = {}
    for r in by_role["val"]:
        by_spk.setdefault(r["speaker"], []).append(r)
    for spk, rs in sorted(by_spk.items()):
        idx = rng_v.permutation(len(rs))[:IND_VAL_PER_SPK]
        VAL_REC[corp] += [rs[i] for i in idx]
    TEST_REC[corp] = sorted(by_role["test"], key=lambda r: (r["speaker"], r["utt"]))
    INTERFERER_REC[corp] = by_role["interferer"]
    TRAIN_REC[corp] = by_role["train"]
    print(f"{corp}: train {len(by_role['train'])} clip / {sum(r['dur'] for r in by_role['train'])/3600:.1f} jam "
          f"({len(set(r['speaker'] for r in by_role['train']))} speaker) | val {len(VAL_REC[corp])} | "
          f"test {len(TEST_REC[corp])} ({len(set(r['speaker'] for r in by_role['test']))} speaker) | "
          f"interferer {len(INTERFERER_REC[corp])} ({len(set(r['speaker'] for r in by_role['interferer']))} speaker)")

In [ ]:
# ==== Turn panjang 10-25 s dari kalimat berurutan satu speaker (mengurangi gap gaya bicara) ====
LT_DIR = "/content/derived/long_turns"
os.makedirs(LT_DIR, exist_ok=True)

def plan_long_turns(clips, rng, target_frac=LONG_TURN_P, min_s=10.0, max_s=25.0, clip_est=3.5):
    # clips: list of record (punya "dur") dalam urutan utterance.
    # target_frac = porsi DATA (bukan porsi grup) yang jadi turn; probabilitas mulai turn
    # dikoreksi karena satu turn mengonsumsi ~clip_est clip: F = q*s/(q*s + 1 - q) -> q = F/(s*(1-F)+F).
    out, i, n = [], 0, len(clips)
    q = float(np.clip(target_frac / (clip_est * (1 - target_frac) + target_frac), 0.0, 1.0))
    while i < n:
        if rng.random() < q:
            grp, tot = [], 0.0
            while i < n and tot < min_s:
                d = clips[i]["dur"]
                if grp and tot + d > max_s:
                    break
                grp.append(clips[i]); tot += d; i += 1
            if tot >= min_s:
                out.append(grp)
            else:  # klip speaker habis sebelum 10 s -> kembalikan sebagai clip tunggal
                out.extend([[g] for g in grp])
        else:
            out.append([clips[i]]); i += 1
    return out

rng_lt = np.random.default_rng(SEED + 5)
_durs_all = [r["dur"] for corp in TRAIN_REC for r in TRAIN_REC[corp]]
_clip_est = float(np.clip(10.0 / max(np.mean(_durs_all), 1.0) + 0.5, 2.0, 7.0))
print(f"Estimasi clip/turn: {_clip_est:.1f} (durasi rata2 {np.mean(_durs_all):.1f} s)")
TURN_REC = {"news": [], "teldialog": []}
for corp, recs in TRAIN_REC.items():
    by_spk = {}
    for r in recs:
        by_spk.setdefault(r["speaker"], []).append(r)
    n_turn = 0
    for spk in sorted(by_spk):
        clips = sorted(by_spk[spk], key=lambda r: r["utt"])
        for k, grp in enumerate(plan_long_turns(clips, rng_lt, clip_est=_clip_est)):
            if len(grp) == 1:
                TURN_REC[corp].append((grp[0], None)); continue
            audio, gap = [], np.zeros(int(0.25 * SR), dtype=np.float32)
            for j, r in enumerate(grp):
                a, sr_ = sf.read(r["path"], dtype="float32")
                if a.ndim > 1: a = a.mean(axis=1)
                audio.append(a)
                if j < len(grp) - 1: audio.append(gap)
            wav = np.concatenate(audio)
            path = f"{LT_DIR}/{corp}_Ind{spk:03d}_{k:03d}.wav"
            sf.write(path, wav, SR, subtype="PCM_16")
            rec = dict(path=path, text=" ".join(r["text"] for r in grp),
                       speaker=spk, corpus=corp, utt=f"turn{k:03d}", dur=len(wav)/SR)
            TURN_REC[corp].append((rec, grp)); n_turn += 1
    print(f"{corp}: {n_turn} turn panjang dibuat")

In [ ]:
# ==== Susun dataset akhir: train / val_clean / TEST_SETS / pool interferer ====
def ds_from_records(recs, is_eval):
    return Dataset.from_dict({
        "audio":   [r["path"] for r in recs],
        "text":    [r["text"] for r in recs],
        "speaker": [f"Ind{r['speaker']:03d}_{r['corpus']}" for r in recs],
        "is_eval": [bool(is_eval)] * len(recs),
    }).cast_column("audio", Audio(sampling_rate=SR))

def finalize_fleurs(ds, is_eval, spk_tag):
    d = ds.select_columns(["audio", "text"]).add_column("speaker", [spk_tag] * len(ds))
    return d.add_column("is_eval", [bool(is_eval)] * len(d))

# FLEURS: buang clip > 30 s, subsample validation, cap test
fleurs_train = fleurs_train.filter(lambda ex: len(ex["audio"]["array"]) <= 30 * SR)
fleurs_test  = fleurs_test.filter(lambda ex: len(ex["audio"]["array"]) <= 30 * SR).select(range(FLEURS_TEST_CAP))
rng_fv = np.random.default_rng(SEED + 6)
fleurs_val = fleurs_val.select(sorted(rng_fv.choice(len(fleurs_val), size=min(FLEURS_VAL_N, len(fleurs_val)), replace=False).tolist()))

# Indspeech train = clip tunggal + turn panjang (dari TURN_REC, bukan clip asli —
# clip asli sudah "dikonsumsi" oleh grup turn). val/test dari speaker hold-out.
ind_train_parts, ind_val_parts = [], []
for corp in ["news", "teldialog"]:
    ind_train_parts.append(ds_from_records([rec for rec, _ in TURN_REC[corp]], False))
    ind_val_parts.append(ds_from_records(VAL_REC[corp], True))

train_ds  = concatenate_datasets([finalize_fleurs(fleurs_train, False, "fleurs")] + ind_train_parts).shuffle(seed=SEED)
val_clean = concatenate_datasets([finalize_fleurs(fleurs_val, True, "fleurs_val")] + ind_val_parts)
TEST_SETS  = {"fleurs_id": finalize_fleurs(fleurs_test, True, "fleurs_test"),
              "indspeech_holdout": concatenate_datasets([ds_from_records(TEST_REC[c], True) for c in ["news", "teldialog"]])}

def decode_pool(recs, n=300, seed=SEED):
    rng = np.random.default_rng(seed)
    pick = rng.permutation(len(recs))[:n]
    pool = []
    for i in pick:
        a, sr_ = sf.read(recs[i]["path"], dtype="float32")
        if a.ndim > 1: a = a.mean(axis=1)
        if sr_ != SR: a = librosa.resample(a, orig_sr=sr_, target_sr=SR)
        pool.append((a.astype(np.float32), f"Ind{recs[i]['speaker']:03d}_{recs[i]['corpus']}"))
    return pool

POOL_TRAIN_RAW = decode_pool([r for corp in ["news", "teldialog"] for r in TRAIN_REC[corp]], 300, SEED + 2)  # interferer utk noise TRAINING (speaker training)
POOL_TEST_RAW  = decode_pool(INTERFERER_REC["news"] + INTERFERER_REC["teldialog"], 300, SEED + 3)  # interferer utk noise TEST

print("train:", len(train_ds), "| val_clean:", len(val_clean),
      "| TEST_SETS:", {k: len(v) for k, v in TEST_SETS.items()},
      "| pool train:", len(POOL_TRAIN_RAW), "| pool test:", len(POOL_TEST_RAW))
summary = dict(train=len(train_ds), val=len(val_clean),
               test={k: len(v) for k, v in TEST_SETS.items()},
               speaker_roles={r: sorted(list(s)) for r, s in ROLES.items()},
               durasi_train_jam=float(sum(rec["dur"] for corp in TURN_REC for rec, _ in TURN_REC[corp]) / 3600 + 9.1))
if os.path.isdir(DRIVE_ROOT):
    with open(f"{DRIVE_ROOT}/data/data_summary.json", "w") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print("Ringkasan data tersimpan di Drive.")

## 3. Feature Extraction MFCC — Manual dari Nol

Pipeline (semua manual pakai NumPy, tanpa `librosa.feature.mfcc`; librosa HANYA dipakai untuk memvalidasi dan visualisasi):

1. **Pre-emphasis** `y[n] = x[n] − 0.97·x[n−1]` → menguatkan frekuensi tinggi (konsonan).
2. **Framing** 25 ms, hop 10 ms → speech dianggap stasioner dalam frame pendek.
3. **Windowing** Hamming → mengurangi spectral leakage.
4. **FFT + power spectrum** `|FFT|² / N` (N = 512).
5. **Mel filterbank** 26 filter segitiga, `mel = 2595·log10(1 + f/700)` → meniru resolusi pendengaran manusia.
6. **Log** energi tiap filter → meniru persepsi loudness & membuat konvolusi channel jadi penjumlahan.
7. **DCT-II** → dekorelasi; ambil koefisien **c0..c12 (13)**.
8. **Liftering** (L = 22) → menyeimbangkan skala koefisien tinggi.

MFCC manual ini TIDAK dimasukkan ke Whisper (Whisper punya log-mel 80-bin internal); MFCC dipakai untuk **VAD (kapan orang bicara)**, **SNR frame speech saat mencampur noise**, dan **embedding speaker** — tiga fungsi yang dibutuhkan notulen "siapa bilang apa".

In [ ]:
def hz_to_mel(f): return 2595.0 * np.log10(1.0 + np.asarray(f) / 700.0)
def mel_to_hz(m): return 700.0 * (10 ** (np.asarray(m) / 2595.0) - 1.0)

def mel_filterbank(n_filters=26, n_fft=512, sr=SR, fmin=0.0, fmax=None):
    fmax = fmax or sr / 2
    hz = mel_to_hz(np.linspace(hz_to_mel(fmin), hz_to_mel(fmax), n_filters + 2))
    fft_freqs = np.linspace(0, sr / 2, n_fft // 2 + 1)
    fb = np.zeros((n_filters, len(fft_freqs)))
    for m in range(n_filters):
        l, c, r = hz[m], hz[m + 1], hz[m + 2]
        up, down = (fft_freqs - l) / (c - l), (r - fft_freqs) / (r - c)
        fb[m] = np.maximum(0, np.minimum(up, down))
    return fb

FB = mel_filterbank()

def mfcc_manual(y, sr=SR, n_mfcc=N_MFCC, frame_ms=25, hop_ms=10, n_fft=512, pre=0.97, lifter=22, return_steps=False):
    y = np.asarray(y, dtype=np.float64)
    emph = np.append(y[0], y[1:] - pre * y[:-1])                       # 1. pre-emphasis
    fl, hop = int(sr * frame_ms / 1000), int(sr * hop_ms / 1000)
    if len(emph) < fl: emph = np.pad(emph, (0, fl - len(emph)))
    n_frames = 1 + (len(emph) - fl) // hop
    idx = np.arange(fl)[None, :] + hop * np.arange(n_frames)[:, None]
    frames = emph[idx]                                                  # 2. framing
    win = frames * np.hamming(fl)                                       # 3. windowing
    power = (np.abs(np.fft.rfft(win, n_fft)) ** 2) / n_fft              # 4. FFT -> power
    fb = FB if (n_fft == 512 and sr == SR) else mel_filterbank(26, n_fft, sr)
    log_mel = np.log(np.maximum(power @ fb.T, 1e-10))                   # 5+6. mel + log
    c = scipy.fftpack.dct(log_mel, type=2, axis=1, norm="ortho")[:, :n_mfcc]   # 7. DCT
    if lifter:                                                          # 8. liftering
        c = c * (1 + (lifter / 2) * np.sin(np.pi * np.arange(n_mfcc) / lifter))
    if return_steps:
        return c, dict(emph=emph, frames=frames, win=win, power=power, log_mel=log_mel)
    return c   # shape: (n_frames, n_mfcc), c0 = log energi frame

In [ ]:
# Validasi vs librosa (hanya untuk mengecek, bukan untuk ekstraksi).
# Setting disamakan: tanpa pre-emphasis & lifter, HTK mel, 26 filter tanpa normalisasi.
y_demo = np.asarray(TEST_SETS["fleurs_id"][0]["audio"]["array"], dtype=np.float32)
fb_lib = librosa.filters.mel(sr=SR, n_fft=512, n_mels=26, htk=True, norm=None)
print("Filterbank manual == librosa:", np.allclose(FB, fb_lib, atol=1e-6))
# librosa menaruh window 400 di tengah frame 512 -> geser 56 sampel; librosa pakai 10·log10 (skala beda, bentuk sama)
m_ours = mfcc_manual(y_demo[56:], pre=0, lifter=0)
S = librosa.feature.melspectrogram(y=y_demo, sr=SR, n_fft=512, hop_length=160, win_length=400, window="hamming",
                                   n_mels=26, center=False, htk=True, power=2.0, norm=None)
m_lib = librosa.feature.mfcc(S=librosa.power_to_db(S, top_db=None), n_mfcc=13).T
n = min(len(m_ours), len(m_lib))
corr = [np.corrcoef(m_ours[:n, i], m_lib[:n, i])[0, 1] for i in range(13)]
print("Korelasi per koefisien (manual vs librosa):", np.round(corr, 4))   # harus ~ 1.0

In [ ]:
# Visualisasi tiap tahap pipeline pada 1 sampel
mf, st = mfcc_manual(y_demo, return_steps=True)
fig, ax = plt.subplots(3, 2, figsize=(14, 10))
t = np.arange(len(y_demo)) / SR
ax[0,0].plot(t, y_demo, lw=.5); ax[0,0].plot(t, st["emph"], lw=.5, alpha=.6); ax[0,0].set_title("Waveform vs pre-emphasis")
k = len(st["frames"]) // 2
ax[0,1].plot(st["frames"][k], label="frame"); ax[0,1].plot(st["win"][k], label="× Hamming"); ax[0,1].legend(); ax[0,1].set_title(f"Frame #{k} (25 ms)")
ax[1,0].plot(10*np.log10(st["power"][k] + 1e-10)); ax[1,0].set_title("Power spectrum (dB) frame tsb")
for f in FB: ax[1,1].plot(f)
ax[1,1].set_title("26 mel filterbank (segitiga)")
ax[2,0].imshow(st["log_mel"].T, aspect="auto", origin="lower"); ax[2,0].set_title("Log-mel energies")
ax[2,1].imshow(mf.T, aspect="auto", origin="lower"); ax[2,1].set_title(f"MFCC c0..c{N_MFCC-1}")
plt.tight_layout(); plt.show()

### VAD dari c0 (energi frame)

c0 = rata-rata log energi mel = **energi/loudness frame**. Aturan floor adaptif (turun cepat, naik pelan — v1) diproses per chunk 0.5 s:

- `p10 = persentil ke-10 c0 chunk` → perkiraan level hening.
- `p10 < floor`: `floor ← 0.7·floor + 0.3·p10` (turun cepat); selain itu `floor ← floor + 0.02·(p10 − floor)` (naik pelan).
- Frame **speech** jika `c0 > floor + margin` (margin 3.0). Run speech < 5 frame (50 ms) dibuang.

Fungsi `chunk_vad` dipakai dua cara: **offline** (`vad_frames`, untuk frame speech saat mencampur noise dan frame voiced untuk embedding speaker) dan **online** (simulasi streaming, Bagian 9: state floor dibawa antar chunk — tidak menghitung ulang fitur seluruh buffer seperti v1).

In [ ]:
VAD_MARGIN, VAD_MIN_FRAMES, VAD_CHUNK_FRAMES = 3.0, 5, int(0.5 / 0.01)  # 50 frame = 0.5 s

def update_floor(floor, p10):
    # Noise floor turun cepat, naik pelan (resep v1 yang dipertahankan).
    if p10 < floor: return 0.7 * floor + 0.3 * p10
    return floor + 0.02 * (p10 - floor)

def chunk_vad(c0_chunk, floor, margin=VAD_MARGIN):
    # VAD satu chunk 0.5 s; kembalikan mask frame + floor baru (state dibawa pemanggil).
    p10 = np.percentile(c0_chunk, 10)
    floor = p10 if floor is None else update_floor(floor, p10)
    return c0_chunk > floor + margin, floor

def _drop_short_runs(mask, min_frames=VAD_MIN_FRAMES):
    out, i = mask.copy(), 0
    while i < len(out):
        if out[i]:
            j = i
            while j < len(out) and out[j]: j += 1
            if j - i < min_frames: out[i:j] = False
            i = j
        else:
            i += 1
    return out

def vad_frames(c0, margin=VAD_MARGIN):
    # Pass offline: proses chunk 0.5 s, floor berjalan, kembalikan mask per frame (10 ms).
    c0 = np.asarray(c0)
    mask = np.zeros(len(c0), dtype=bool); floor = None
    for s in range(0, len(c0), VAD_CHUNK_FRAMES):
        ch = c0[s:s + VAD_CHUNK_FRAMES]
        m, floor = chunk_vad(ch, floor, margin)
        mask[s:s + len(ch)] = m
    return _drop_short_runs(mask)

def voiced_mfcc(y, margin=VAD_MARGIN):
    # MFCC hanya pada frame voiced (dipakai embedding speaker Bagian 5 & 10).
    mf = mfcc_manual(y)
    m = vad_frames(mf[:, 0], margin)
    if m.sum() < VAD_MIN_FRAMES: m = np.ones(len(mf), dtype=bool)  # fallback: semua frame
    return mf[m]

c0_demo = mfcc_manual(y_demo)[:, 0]; v_demo = vad_frames(c0_demo)
fig, ax = plt.subplots(2, 1, figsize=(14, 4), sharex=True)
ax[0].plot(np.arange(len(y_demo)) / SR, y_demo, lw=.5); ax[0].set_title("Waveform")
tt = np.arange(len(c0_demo)) * 0.01
ax[1].plot(tt, c0_demo, label="c0"); ax[1].fill_between(tt, c0_demo.min(), c0_demo.max(), where=v_demo, alpha=.2, label="speech (VAD)")
ax[1].legend(); plt.tight_layout(); plt.show()

## 4. Bank Noise + `add_noise`

Tiga jenis noise, semuanya **dipisah menjadi bagian training dan bagian test** supaya noise uji tidak pernah dilihat training:

| Jenis | Sumber | Bagian training | Bagian test |
|---|---|---|---|
| `env` (lingkungan) | MUSAN subset noise (929 file ~6 jam, 16 kHz, CC BY 4.0 — cek lisensi per folder) + DEMAND 16 kHz (17 environment) | 80% file MUSAN + 14 environment DEMAND | 20% file MUSAN + OMEETING/OOFFICE/OHALLWAY (paling mirip ruang meeting) |
| `babble` (orang lain bicara) | sintesis 4–8 clip speech Indonesia dari pool interferer, offset acak, speaker berbeda bila memungkinkan | 300 clip dari speaker training | 300 clip dari 10+10 speaker interferer hold-out |
| `overlap` (tumpang tindih) | 1–2 clip interferer yang menutupi >= 50% durasi target | idem | idem |

**Definisi SNR (di frame speech):** `10·log10(kuasa target di frame speech ÷ kuasa noise di frame yang sama)` — frame speech didapat dari `vad_frames` pada target bersih. Kalau puncak campuran > 1.0, seluruh campuran diskalakan turun (aman: log-mel Whisper invarian terhadap gain keseluruhan, extractor Wav2Vec2 menormalisasi zero-mean unit-variance).

Lisensi DEMAND tercantum CC BY-SA 3.0 di situsnya dan CC BY 4.0 di metadata Zenodo — **keduanya disitasi** di laporan.

In [ ]:
# ==== Download MUSAN (subset noise) + DEMAND 16k (ch01 tiap environment) ====
NOISE_DIR = "/content/noise"
MUSAN_URLS = ["https://www.openslr.org/resources/17/musan.tar.gz",
              "https://us.openslr.org/resources/17/musan.tar.gz"]
DEMAND_URL = "https://zenodo.org/records/1227121/files/{env}_16k.zip"
DEMAND_ENVS = ["DKITCHEN", "DLIVING", "DWASHING", "NFIELD", "NPARK", "NRIVER", "OHALLWAY",
               "OMEETING", "OOFFICE", "PCAFETER", "PRESTO", "PSTATION", "SPSQUARE",
               "STRAFFIC", "TBUS", "TCAR", "TMETRO"]   # 17 environment 16 kHz (SCAFE tidak ada versi 16k)
DEMAND_TEST_ENVS = ["OMEETING", "OOFFICE", "OHALLWAY"]  # hold-out: ruangan kantor/meeting = domain deploy

def _wget(url, path, tries=3):
    for _ in range(tries):
        try:
            urllib.request.urlretrieve(url, path)
            if os.path.getsize(path) > 1_000_000: return True
        except Exception:
            time.sleep(2)
    return False

def ensure_noise_data():
    os.makedirs(NOISE_DIR, exist_ok=True)
    # MUSAN: tarball ~11 GB berisi musan/{music,speech,noise}; kita stream-extract HANYA musan/noise (~700 MB).
    musan_noise = f"{NOISE_DIR}/musan/noise"
    if not (os.path.isdir(musan_noise) and sum(len(fs) for _, _, fs in os.walk(musan_noise)) > 900):
        drive_musan = f"{DRIVE_ROOT}/data/musan_noise"
        if CACHE_DATA_TO_DRIVE and os.path.isdir(drive_musan) and \
                sum(len(fs) for _, _, fs in os.walk(drive_musan)) > 900:
            shutil.copytree(drive_musan, musan_noise, dirs_exist_ok=True)
            print("MUSAN noise: restore dari cache Drive")
        else:
            print("MUSAN: stream-extract musan/noise dari tarball 11 GB (3-6 menit)...")
            ok = False
            for url in MUSAN_URLS:
                rc = os.system(f"wget -qO- '{url}' | tar -xz -C '{NOISE_DIR}' musan/noise")
                if rc == 0 and os.path.isdir(musan_noise): ok = True; break
            assert ok, "Download MUSAN gagal - cek koneksi / jalankan ulang cell"
            if CACHE_DATA_TO_DRIVE and os.path.isdir("/content/drive/MyDrive"):
                shutil.copytree(musan_noise, drive_musan, dirs_exist_ok=True)
                print("MUSAN noise: dicache ke Drive")
    # DEMAND 16k: unduh 17 zip (~1.8 GB total), ekstrak HANYA ch01.wav (~9.6 MB/environment).
    os.makedirs(f"{NOISE_DIR}/demand", exist_ok=True)
    drive_ch01 = f"{DRIVE_ROOT}/data/demand_ch01"
    if CACHE_DATA_TO_DRIVE and os.path.isdir("/content/drive/MyDrive"):
        os.makedirs(drive_ch01, exist_ok=True)
    for env in DEMAND_ENVS:
        dst = f"{NOISE_DIR}/demand/{env}_ch01.wav"
        if os.path.exists(dst): continue
        dcache = f"{drive_ch01}/{env}_ch01.wav"
        if os.path.exists(dcache):                    # restore ch01 dari cache Drive (cepat)
            shutil.copy(dcache, dst); continue
        zp = f"{NOISE_DIR}/{env}_16k.zip"
        if not os.path.exists(zp):
            print(f"DEMAND {env}: unduh...")
            assert _wget(DEMAND_URL.format(env=env), zp), f"Gagal unduh DEMAND {env}"
        with zipfile.ZipFile(zp) as z:
            names = [n for n in z.namelist() if "ch01" in n and n.endswith(".wav")]
            with open(dst, "wb") as f: f.write(z.read(names[0]))
        os.remove(zp)
        if CACHE_DATA_TO_DRIVE and os.path.isdir("/content/drive/MyDrive"):
            shutil.copy(dst, dcache)                   # cache ch01 ke Drive
    print("File noise:", sorted(os.listdir(f"{NOISE_DIR}/demand")))

ensure_noise_data()
n_musan = sum(len(fs) for _, _, fs in os.walk(f"{NOISE_DIR}/musan/noise"))
lic = [p for p in glob.glob(f"{NOISE_DIR}/musan/noise/**/*", recursive=True)
       if os.path.isfile(p) and ("LICEN" in p.upper() or p.upper().endswith(".TXT"))]
print(f"MUSAN noise: {n_musan} file | file lisensi yang ditemukan: {lic[:5]} (catat utk laporan)")

In [ ]:
# ==== Bangun bank noise: env (path file) + babble/overlap (pool clip in-memory) ====
import glob

def build_noise_banks(seed=SEED):
    rng = np.random.default_rng(seed)
    musan_files = sorted(glob.glob(f"{NOISE_DIR}/musan/noise/**/*.wav", recursive=True))
    musan_files = [p for p in musan_files if "LICEN" not in p.upper()]
    rng.shuffle(musan_files)
    k = int(0.8 * len(musan_files))
    env_train = musan_files[:k] + [f"{NOISE_DIR}/demand/{e}_ch01.wav" for e in DEMAND_ENVS if e not in DEMAND_TEST_ENVS]
    env_test  = musan_files[k:] + [f"{NOISE_DIR}/demand/{e}_ch01.wav" for e in DEMAND_TEST_ENVS]
    return {"train": {"env": env_train, "babble": POOL_TRAIN_RAW},
            "test":  {"env": env_test,  "babble": POOL_TEST_RAW}}

NOISE_BANK = build_noise_banks()
print("Bank noise | train: env", len(NOISE_BANK["train"]["env"]), "file, babble pool",
      len(NOISE_BANK["train"]["babble"]), "clip,", len(set(s for _, s in NOISE_BANK["train"]["babble"])), "speaker")
print("           | test : env", len(NOISE_BANK["test"]["env"]), "file, babble pool",
      len(NOISE_BANK["test"]["babble"]), "clip,", len(set(s for _, s in NOISE_BANK["test"]["babble"])), "speaker")
print("           | env test:", [os.path.basename(p) for p in NOISE_BANK['test']['env'] if 'demand' in p])

In [ ]:
# ==== add_noise: campur noise sesuai jenis + SNR pada frame speech ====
HOP_S, FRAME_S = 160, 400   # hop 10 ms, frame 25 ms (sama dengan mfcc_manual)

def expand_frame_mask(mask, n_samples, hop=HOP_S, fl=FRAME_S):
    out = np.zeros(n_samples, dtype=bool)
    for s in np.flatnonzero(mask):
        out[s * hop: s * hop + fl] = True
    return out

def speech_power(x, sample_mask):
    if sample_mask.sum() == 0: sample_mask = np.ones(len(x), dtype=bool)
    return float(np.mean(np.asarray(x, dtype=np.float64)[sample_mask] ** 2))

def _env_noise(path, n, rng):
    info = sf.info(path)
    if info.samplerate != SR:   # jarang (MUSAN/DEMAND 16k), tapi jaga-jaga
        a, sr_ = sf.read(path, dtype="float32")
        if a.ndim > 1: a = a.mean(axis=1)
        a = librosa.resample(a, orig_sr=sr_, target_sr=SR)
        start = int(rng.integers(0, max(1, len(a) - n))) if len(a) > n else 0
        seg = np.concatenate([a[start:], a])[:n]
        return seg.astype(np.float32)
    start = int(rng.integers(0, max(1, info.frames - n))) if info.frames > n else 0
    seg, _ = sf.read(path, start=start, frames=min(n, info.frames), dtype="float32")
    if seg.ndim > 1: seg = seg.mean(axis=1)
    while len(seg) < n:
        seg = np.concatenate([seg, seg[:n - len(seg)]])
    return seg[:n].astype(np.float32)

def _pick_distinct(pool, k, rng):
    # pilih k clip dengan speaker sebeda mungkin
    order = rng.permutation(len(pool))
    out, seen = [], set()
    for i in order:
        a, spk = pool[int(i)]
        if spk not in seen:
            seen.add(spk); out.append((a, spk))
            if len(out) == k: return out
    for i in order:                    # pool speaker < k -> ulang speaker sisanya
        out.append(pool[int(i)])
        if len(out) == k: return out
    return out

def _cyclic_from(a, n, rng):
    o = int(rng.integers(0, max(1, len(a))))   # offset acak
    return np.concatenate([a[o:], a, a])[:n]

def _babble_noise(pool, n, rng):
    # 4-8 pembicara lain, offset acak, jumlahinya (karakter babble makin padat).
    k = int(rng.integers(4, 9))
    chosen = _pick_distinct(pool, k, rng)
    noise = np.zeros(n, dtype=np.float64)
    for a, _ in chosen:
        noise += _cyclic_from(a, n, rng)
    return noise.astype(np.float32)

def _overlap_noise(pool, n, rng):
    # 1-2 interferer yang menutupi >= 50% durasi target, ditempatkan berdampingan.
    a0, _ = pool[int(rng.integers(len(pool)))]
    w0 = min(len(a0), n)
    if w0 >= n / 2:
        s = int(rng.integers(0, max(1, n - w0 + 1)))
        out = np.zeros(n, dtype=np.float32); out[s:s + w0] = a0[:w0]
        return out
    w1_need = n / 2 - w0
    a1, _ = pool[int(rng.integers(len(pool)))]
    for _ in range(10):                 # cari interferer ke-2 yang cukup panjang
        if min(len(a1), n) >= w1_need: break
        a1, _ = pool[int(rng.integers(len(pool)))]
    w1 = min(len(a1), n)
    out = np.zeros(n, dtype=np.float32)
    if w0 + w1 >= n:
        out[:w0] = a0[:w0]; out[w0:n] = a1[:n - w0]
        return out
    s = int(rng.integers(0, max(1, n - (w0 + w1) + 1)))
    out[s:s + w0] = a0[:w0]; out[s + w0:s + w0 + w1] = a1[:w1]
    return out

def add_noise(audio, kind, snr_db, rng, bank):
    """Campur noise ke audio bersih. kind: 'babble'|'overlap'|'env'. bank: bank train ATAU test."""
    audio = np.asarray(audio, dtype=np.float32)
    n = len(audio)
    if kind == "env":
        noise = _env_noise(bank["env"][int(rng.integers(len(bank["env"])))], n, rng)
    elif kind == "babble":
        noise = _babble_noise(bank["babble"], n, rng)
    elif kind == "overlap":
        noise = _overlap_noise(bank["babble"], n, rng)
    else:
        raise ValueError(kind)
    mask_f = vad_frames(mfcc_manual(audio)[:, 0])
    m = expand_frame_mask(mask_f, n)
    ps, pn = speech_power(audio, m), speech_power(noise, m)
    scale = np.sqrt(ps / max(pn, 1e-12) / 10 ** (snr_db / 10.0))
    mix = audio + noise * scale
    peak = float(np.abs(mix).max())
    if peak > 1.0:
        mix = mix / peak   # log-mel Whisper & extractor w2v2 invarian gain keseluruhan
    return mix.astype(np.float32)

def parse_cond(cond):
    if cond == "clean": return None, None
    kind, snr = cond.rsplit("_", 1)
    return kind, float(snr)

def cond_seed(name, cond):
    return int(zlib.crc32(f"{name}|{cond}".encode()) & 0xFFFFFFFF)   # deterministik antar sesi

CONDITIONS = ["clean"] + [f"{k}_{s}" for k in NOISE_KINDS for s in SNR_TEST]
if INCLUDE_SNR0:
    CONDITIONS += [f"{k}_0" for k in NOISE_KINDS]
print("Kondisi evaluasi:", CONDITIONS)

def make_noisy_set(ds, cond, seed, bank):
    """Versi noise dari suatu dataset; DETERMINISTIK: satu seed per (test set, kondisi)."""
    rng = np.random.default_rng(seed)
    kind, snr = parse_cond(cond)
    audios, refs = [], []
    for item in ds:
        a = np.asarray(item["audio"]["array"], dtype=np.float32)
        if kind is not None:
            a = add_noise(a, kind, snr, rng, bank)
        audios.append(a); refs.append(item["text"])
    return audios, refs

In [ ]:
# ==== Self-check add_noise (dijalankan sebelum dipakai training/eval) ====
# 1) SNR yang benar-benar tercapai di frame speech (dihitung ulang dari sinyal sisa mix - target)
rng_chk = np.random.default_rng(0)
_t = np.zeros(int(4 * SR), np.float32)
for s in range(4):                       # target sintetis: burst "speech" + hening, puncak 0.3 (bebas clipping)
    _t[int(s * SR + 0.2 * SR): int(s * SR + 0.7 * SR)] = 0.3 * np.sin(2 * np.pi * 220 * np.arange(int(0.5 * SR)) / SR)
_fake_pool = [(0.05 * np.sin(2 * np.pi * (100 + 30 * i) * np.arange(int(3 * SR)) / SR).astype(np.float32), f"spk{i}") for i in range(6)]
_fake_bank = {"env": [], "babble": _fake_pool}
mix = add_noise(_t, "babble", 10.0, rng_chk, _fake_bank)
resid = mix - _t
_m = expand_frame_mask(vad_frames(mfcc_manual(_t)[:, 0]), len(_t))
snr_meas = 10 * np.log10(speech_power(_t, _m) / max(speech_power(resid, _m), 1e-12))
print(f"SNR target 10 dB -> terukur {snr_meas:.2f} dB (wajib dekat 10, toleransi +-0.3)")
# 2) Determinisme: seed sama -> hasil identik
r1, r2 = np.random.default_rng(7), np.random.default_rng(7)
a1 = add_noise(_t, "babble", 5.0, r1, _fake_bank); a2 = add_noise(_t, "babble", 5.0, r2, _fake_bank)
print("Deterministik:", np.array_equal(a1, a2))
# 3) Overlap menutupi >= 50% target
cov = []
for _ in range(20):
    nn = _overlap_noise(_fake_pool, len(_t), np.random.default_rng(np.random.randint(1 << 30)))
    cov.append((np.abs(nn) > 0).mean())
print(f"Coverage overlap: min {min(cov)*100:.0f}% (wajib >= 50%), mean {np.mean(cov)*100:.0f}%")

In [ ]:
# ==== Visualisasi: satu clip bersih vs ketiga jenis noise @ 10 dB ====
_clip = TEST_SETS["indspeech_holdout"][0]
_a = np.asarray(_clip["audio"]["array"], np.float32)
fig, ax = plt.subplots(4, 1, figsize=(13, 10))
for i, kind in enumerate([None] + NOISE_KINDS):
    a = _a if kind is None else add_noise(_a, kind, 10.0, np.random.default_rng(123), NOISE_BANK["test"])
    ax[i].plot(np.arange(len(a)) / SR, a, lw=.4)
    ax[i].set_title("bersih" if kind is None else f"{kind} @ 10 dB")
plt.tight_layout(); plt.show()
print("Referensi:", _clip["text"])

## 5. Eksperimen 13 vs 12 Koefisien MFCC (speaker ID + verifikasi, kondisi keras)

v1 membandingkan 12 vs 13 koefisien lewat speaker identification pada 30 clip LibriSpeech Inggris dan keduanya mencapai akurasi 1.000 — langit-langit, tanpa kesimpulan. v2 mendesain ulang:

- **Speaker**: 20-30 speaker **hold-out** Indspeech (validasi + test, tidak pernah dilihat training), masing-masing >= 8 clip (cap 12 clip biar seimbang).
- **Identifikasi**: GMM diagonal per speaker (gaya GMM-UBM, sesuai materi HMM-GMM), frame voiced.
- **Verifikasi**: pasangan same-speaker vs different-speaker (ratusan pasangan), skor **cosine** pada embedding Bagian 10 (mean+std MFCC frame voiced, z-normalisasi memakai statistik dari speaker kalibrasi — two-fold supaya semua speaker ke-skor). Metrik: **equal error rate (EER)**.
- **Kondisi keras**: bersih, **babble 10 dB** (interferer dari pool test — speaker berbeda), dan **clip 2 detik**.

Yang dibandingkan: fitur `c1..c12` (tanpa c0) vs `c0..c12`. VAD selalu memakai c0 — jadi hasil sah yang diharapkan bisa dua arah: "c0 tidak membantu speaker ID tapi dibutuhkan untuk VAD" pun merupakan temuan yang valid untuk notulen.

In [ ]:
# ==== Susun data speaker utk eksperimen (hold-out, >= 8 clip) ====
rng_e = np.random.default_rng(SEED + 10)
SPEAKER_DATA = {}
for corp in ["news", "teldialog"]:
    for r in ALL_REC[corp]:
        if r["speaker"] in ROLES["val"] or r["speaker"] in ROLES["test"]:
            SPEAKER_DATA.setdefault(f"Ind{r['speaker']:03d}_{corp}", []).append(r)
for k in list(SPEAKER_DATA):
    rs = SPEAKER_DATA[k]
    idx = rng_e.permutation(len(rs))[:12]
    SPEAKER_DATA[k] = [np.asarray(sf.read(rs[i]["path"], dtype="float32")[0], dtype=np.float32) for i in idx]
SPEAKER_DATA = {k: v for k, v in SPEAKER_DATA.items() if len(v) >= 8}
print(f"Speaker eksperimen: {len(SPEAKER_DATA)} (target 20-30), clip/speaker: "
      f"min {min(len(v) for v in SPEAKER_DATA.values())}, max {max(len(v) for v in SPEAKER_DATA.values())}")

In [ ]:
# ==== Identifikasi (GMM diagonal) + verifikasi (cosine + EER), 2 set fitur x 3 kondisi ====
from sklearn.mixture import GaussianMixture

def eer_from_scores(scores, labels):
    # labels: 1 = same speaker. Sweep threshold; EER = titik FAR == FRR.
    scores = np.asarray(scores); labels = np.asarray(labels)
    order = np.argsort(-scores)
    srt = labels[order]
    P, N = max(labels.sum(), 1), max((1 - labels).sum(), 1)
    tpr = np.cumsum(srt) / P            # same-speaker diterima
    fpr = np.cumsum(1 - srt) / N        # diff-speaker lolos
    fnr = 1 - tpr
    i = int(np.argmin(np.abs(fnr - fpr)))
    return float((fnr[i] + fpr[i]) / 2)

def speaker_embed(mf_voiced, mu, sd):
    z = (mf_voiced - mu) / (sd + 1e-6)
    e = np.concatenate([z.mean(0), z.std(0)])
    return e / (np.linalg.norm(e) + 1e-9)   # normalisasi supaya dot = cosine

def feats_by_cond(cond):
    # kembalikan {speaker: [MFCC voiced 13 kolom per clip]} untuk satu kondisi
    out = {}
    for k, clips in SPEAKER_DATA.items():
        A = clips
        if cond == "clip2s":
            A = [a[:int(2.0 * SR)] for a in A]
        if cond == "babble_10":
            rng = np.random.default_rng(cond_seed(k, cond))
            A = [add_noise(a, "babble", 10.0, rng, NOISE_BANK["test"]) for a in A]
        out[k] = [voiced_mfcc(a) for a in A]
    return out

def speaker_id_acc(feats):
    models, tests = {}, []
    for k, F in feats.items():
        ntr = max(2, int(0.6 * len(F)))
        models[k] = GaussianMixture(8, covariance_type="diag", random_state=SEED,
                                    reg_covar=1e-3).fit(np.vstack(F[:ntr]))
        tests += [(k, f) for f in F[ntr:]]
    return float(np.mean([max(models, key=lambda m: models[m].score(f)) == k for k, f in tests])), len(tests)

def verify_eer(feats):
    keys = sorted(feats); half = len(keys) // 2
    scores, labels = [], []
    for cal, sc in [(keys[:half], keys[half:]), (keys[half:], keys[:half])]:
        calF = np.vstack([f for k in cal for f in feats[k]])
        mu, sd = calF.mean(0), calF.std(0)
        emb = {k: [speaker_embed(f, mu, sd) for f in feats[k]] for k in sc}
        rngp = np.random.default_rng(SEED + 77)
        for k in sc:
            E = emb[k]
            for i in range(len(E)):
                for j in range(i + 1, len(E)):
                    scores.append(float(E[i] @ E[j])); labels.append(1)
        pairs = [(k1, k2) for k1 in sc for k2 in sc if k1 < k2]
        for i in rngp.permutation(len(pairs))[:len(pairs)]:
            k1, k2 = pairs[i]
            for _ in range(8):   # beberapa pasangan clip per pasangan speaker -> seimbang dgn pasangan same
                e1 = emb[k1][int(rngp.integers(len(emb[k1])))]; e2 = emb[k2][int(rngp.integers(len(emb[k2])))]
                scores.append(float(e1 @ e2)); labels.append(0)
    return eer_from_scores(scores, labels), sum(1 for l in labels if l == 1), sum(1 for l in labels if l == 0)

rows13 = []
for cond in ["clean", "babble_10", "clip2s"]:
    feats13 = feats_by_cond(cond)
    for name, sl in {"c0-c12 (13)": slice(0, 13), "c1-c12 (12)": slice(1, 13)}.items():
        feats = {k: [f[sl] for f in v] for k, v in feats13.items()}
        acc, n_id = speaker_id_acc(feats)
        eer, n_same, n_diff = verify_eer(feats)
        rows13.append(dict(kondisi=cond, fitur=name, akurasi_ID=round(acc, 3), EER=round(eer, 3),
                           n_uji_ID=n_id, n_pasangan_same=n_same, n_pasangan_diff=n_diff))
        print(f"{cond:10s} {name}: akurasi ID {acc:.3f} | EER {eer:.3f} (same {n_same}, diff {n_diff})")
df13 = pd.DataFrame(rows13)
display(df13.pivot_table(index="kondisi", columns="fitur", values=["akurasi_ID", "EER"]))
if os.path.isdir(DRIVE_ROOT):
    df13.to_csv(f"{DRIVE_ROOT}/eval/eksperimen_13v12.csv", index=False)
print("Catatan utk laporan: c0 tetap dipakai VAD di semua kondisi; perbandingan ini hanya tentang fitur speaker.")

## 6. Baseline: Whisper Pretrained + Harness Evaluasi + Tuning Argumen Decode

Metrik utama **WER** (skor `norm()`: huruf kecil tanpa tanda baca — satu normalisasi sama untuk semua sistem, karena kelakannya akan dibandingkan dengan CTC yang memang tanpa casing/punctuation), **CER**, dan **RTF** (waktu proses / durasi audio; < 1 = lebih cepat dari real-time).

**Argumen anti-halusinasi v1** (`repetition_penalty=1.2`, `no_repeat_ngram_size=3`, `no_speech_threshold=0.6`, `logprob_threshold=-1.0`) tidak pernah di-benchmark -> v2 mengukurnya **dengan vs tanpa** pada validation **bersih dan babble 10 dB** (noise dari bank TRAINING — bank test tidak boleh disentuh sebelum pengujian akhir), lalu menyimpan yang menang. Tidak pernah di-tune di test. `temperature=0.0` wajib di transformers 4.46.3 jika `logprob_threshold` aktif (bug `NoneType > float` di `_retrieve_avg_logprobs`) — 0.0 tetap greedy, hanya menambal library.

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

normalizer = BasicTextNormalizer()
def norm(t): return " ".join(normalizer(t).split())

GEN_KW_PLAIN = dict(task="transcribe", max_new_tokens=220, temperature=0.0)
GEN_KW_ANTI  = dict(GEN_KW_PLAIN, repetition_penalty=1.2, no_repeat_ngram_size=3,
                    no_speech_threshold=0.6, logprob_threshold=-1.0)

def set_generation_config(mdl, gen_kw, language=LANGUAGE):
    # Trainer men-generate dari model.generation_config, BUKAN dari argumen evaluate kita;
    # samakan keduanya supaya checkpoint terbaik dipilih dengan decoder yang sama dengan
    # yang dilaporkan (syarat spesifikasi v2). forced_decoder_ids=None: bahasa/task lewat
    # argumen generate (cara modern, tanpa warning bentrok).
    g = mdl.generation_config
    g.language, g.task, g.forced_decoder_ids = language, "transcribe", None
    mdl.config.forced_decoder_ids = None
    for k, v in dict(gen_kw, language=language).items():
        setattr(g, k, v)

def load_whisper(name, gen_kw=None):
    proc = WhisperProcessor.from_pretrained(name)
    mdl = WhisperForConditionalGeneration.from_pretrained(name).to(DEVICE).eval()
    set_generation_config(mdl, gen_kw or GEN_KW_PLAIN)
    if DEVICE == "cuda": mdl = mdl.half()
    return proc, mdl

@torch.no_grad()
def transcribe(proc, mdl, audios, gen_kw=None, language=LANGUAGE, bs=BS_EVAL):
    out, base = [], dict(gen_kw or GEN_KW_PLAIN)
    if language: base["language"] = language
    for i in range(0, len(audios), bs):
        feat = proc.feature_extractor(audios[i:i+bs], sampling_rate=SR, return_tensors="pt")
        kw = dict(base)
        if "attention_mask" in feat:
            kw["attention_mask"] = feat["attention_mask"].to(DEVICE)
        ids = mdl.generate(feat.input_features.to(DEVICE, mdl.dtype), **kw)
        out += proc.batch_decode(ids, skip_special_tokens=True)
    return [o.strip() for o in out]

def clip_error_counts(R, H):
    # per clip: (sub, del, ins, panjang_ref = hit+sub+del) -> dipakai bootstrap CI
    out = []
    for r, h in zip(R, H):
        p = jiwer.process_words(r, h)
        out.append((p.substitutions, p.deletions, p.insertions, p.substitutions + p.deletions + p.hits))
    return np.asarray(out, dtype=np.float64)

def bootstrap_ci(counts, n=1000, seed=SEED):
    # CI 95% WER: resample clip 1000x, WER = (S+D+I)/N_ref dari jumlahan resample.
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(counts), size=(n, len(counts)))
    s = counts[idx].sum(axis=1)
    wers = (s[:, 0] + s[:, 1] + s[:, 2]) / np.maximum(s[:, 3], 1.0)
    return tuple(float(x) for x in np.percentile(wers, [2.5, 97.5]))

def evaluate(proc, mdl, audios, refs, gen_kw=None, bs=BS_EVAL):
    # v2: menerima audio + referensi yang SUDAH siap (noise diterapkan di luar) sehingga
    # noise identik antar model dan hasil eval murni fungsi model.
    refs_n = [norm(t) for t in refs]
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    hyps = transcribe(proc, mdl, audios, gen_kw, bs=bs)
    if DEVICE == "cuda": torch.cuda.synchronize()
    dt, dur = time.time() - t0, sum(len(a) for a in audios) / SR
    hyps_n = [norm(h) for h in hyps]
    keep = [i for i, r in enumerate(refs_n) if r]
    R, H = [refs_n[i] for i in keep], [hyps_n[i] for i in keep]
    counts = clip_error_counts(R, H)
    wer = float((counts[:, 0].sum() + counts[:, 1].sum() + counts[:, 2].sum()) / max(counts[:, 3].sum(), 1))
    return dict(wer=wer, cer=float(jiwer.cer(R, H)), rtf=dt / max(dur, 1e-9), counts=counts), (R, H, [hyps[i] for i in keep])

def build_val_audios(force=False):
    # Validation utk tuning: bersih + babble 10 dB (noise dari bank TRAINING, deterministik).
    global VAL_CLEAN_AUDIO, VAL_B10_AUDIO, VAL_REFS
    _cached = globals().get("VAL_CLEAN_AUDIO")
    if _cached is not None and not force:
        return _cached, globals().get("VAL_B10_AUDIO"), globals().get("VAL_REFS")
    VAL_CLEAN_AUDIO, VAL_REFS = make_noisy_set(val_clean, "clean", cond_seed("val", "clean"), NOISE_BANK["train"])
    VAL_B10_AUDIO, _ = make_noisy_set(val_clean, "babble_10", cond_seed("val", "babble_10"), NOISE_BANK["train"])
    return VAL_CLEAN_AUDIO, VAL_B10_AUDIO, VAL_REFS

In [ ]:
# ==== Tuning argumen decode (7.5): dengan vs tanpa anti-halusinasi, HANYA di validation ====
GEN_KW_CACHE = f"{DRIVE_ROOT}/logs/gen_kw_tuning.json"
GEN_KW_BY_MODEL = {}
if os.path.exists(GEN_KW_CACHE):
    GEN_KW_BY_MODEL = {k: GEN_KW_ANTI if v == "anti" else GEN_KW_PLAIN
                       for k, v in json.load(open(GEN_KW_CACHE)).items()}
    print("Cache tuning dimuat:", json.load(open(GEN_KW_CACHE)))

vc, vn, vr = build_val_audios()
tune_rows = []
for name in [m for m in ["openai/whisper-tiny", "openai/whisper-base"] if m not in GEN_KW_BY_MODEL]:
    proc, mdl = load_whisper(name)
    w = {}
    for gname, gk in [("plain", GEN_KW_PLAIN), ("anti", GEN_KW_ANTI)]:
        m_c, _ = evaluate(proc, mdl, vc, vr, gk)
        m_n, _ = evaluate(proc, mdl, vn, vr, gk)
        w[gname] = (m_c["wer"] + m_n["wer"]) / 2
        tune_rows.append(dict(model=name.split("/")[-1], config=gname,
                              wer_val_bersih=round(m_c["wer"], 4), wer_val_babble10=round(m_n["wer"], 4),
                              rata2=round(w[gname], 4)))
    win = min(w, key=w.get)
    GEN_KW_BY_MODEL[name] = GEN_KW_PLAIN if win == "plain" else GEN_KW_ANTI
    print(f"{name}: pemenang = {win}")
    del mdl; torch.cuda.empty_cache()
if tune_rows:
    display(pd.DataFrame(tune_rows))
if os.path.isdir(DRIVE_ROOT) and not os.path.exists(GEN_KW_CACHE):
    with open(GEN_KW_CACHE, "w") as f:
        json.dump({k: "anti" if v is GEN_KW_ANTI else "plain" for k, v in GEN_KW_BY_MODEL.items()}, f)
GEN_KW_KEPT_BASE = GEN_KW_BY_MODEL["openai/whisper-base"]
print("Argumen decode whisper-base:", "ANTI" if GEN_KW_KEPT_BASE is GEN_KW_ANTI else "PLAIN")

In [ ]:
# ==== Baseline bersih (smoke test harness sebelum training; grid lengkap ada di Bagian 8) ====
rows_b = []
audios_f, refs_f = make_noisy_set(TEST_SETS["fleurs_id"], "clean", 0, None)
for name in ["openai/whisper-tiny", "openai/whisper-base"]:
    proc, mdl = load_whisper(name, GEN_KW_BY_MODEL[name])
    m, _ = evaluate(proc, mdl, audios_f, refs_f)
    lo, hi = bootstrap_ci(m["counts"])
    rows_b.append(dict(model=name.split("/")[-1], wer=round(m["wer"], 4), ci=f"[{lo:.3f},{hi:.3f}]",
                      cer=round(m["cer"], 4), rtf=round(m["rtf"], 4)))
    print(f"{name:20s} fleurs_id bersih: WER={m['wer']:.3f} [{lo:.3f},{hi:.3f}] CER={m['cer']:.3f} RTF={m['rtf']:.3f}")
    del mdl; torch.cuda.empty_cache()
df_b = pd.DataFrame(rows_b)
if os.path.isdir(DRIVE_ROOT): df_b.to_csv(f"{DRIVE_ROOT}/eval/baseline_bersih.csv", index=False)

## 7. Fine-tune Whisper-base + Ablation Clean-Only

| | Whisper-base (utama) | Ablation |
|---|---|---|
| Titik awal | `openai/whisper-base` (74M) | sama |
| p_noise | **0.6** (jenis uniform, SNR uniform 0-20 dB, on-the-fly di collator) | **0** (bersih semua) |
| lr / batch / step | 1e-5 / 16 / MAX_STEPS=1500 | sama persis |
| Validation checkpoint | val_clean (bersih, speaker hold-out, `is_eval`=True -> tidak pernah di-augmentasi) | sama |
| Tujuan | model utama deploy browser | bukti (b): porsi WER di noise yang berasal dari augmentasi |

Ketentuan fairness: dataset, augmentasi (kecuali p_noise), budget step, dan evaluasi identik; `generation_config` disamakan dengan argumen decode pemenang Bagian 6 supaya pemilihan checkpoint dan laporan memakai decoder yang sama. Noise augmentasi hanya dari **bank training**. Setiap model langsung di-backup ke Drive begitu training selesai.

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

class Collator:
    # Augmentasi HANYA untuk baris is_eval=False (perbaikan v1: validation tidak pernah
    # ter-augmentasi). Kolom selain input_features/labels tidak diteruskan ke model.
    def __init__(self, processor, decoder_start_token_id, p_noise=0.6, bank=None, seed=SEED):
        self.processor, self.dsid = processor, decoder_start_token_id
        self.p_noise, self.bank, self.seed = p_noise, bank, seed
        self._gen = None
    def _rng(self):
        # Seed per dataloader-worker (workers=2); tanpa ini dua worker menghasilkan noise identik.
        if self._gen is None:
            info = torch.utils.data.get_worker_info()
            wid = info.id if info is not None else 0
            self._gen = np.random.default_rng(self.seed + 9973 * wid)
        return self._gen
    def __call__(self, batch):
        g = self._rng()
        audios = []
        for b in batch:
            a = np.asarray(b["audio"]["array"], dtype=np.float32)
            if self.p_noise > 0 and not bool(b.get("is_eval", False)) and g.random() < self.p_noise:
                kind = NOISE_KINDS[int(g.integers(len(NOISE_KINDS)))]
                snr = float(g.uniform(SNR_LO, SNR_HI))
                a = add_noise(a, kind, snr, g, self.bank)
            audios.append(a)
        feats = self.processor.feature_extractor(audios, sampling_rate=SR, return_tensors="pt").input_features
        lab = self.processor.tokenizer([b["text"] for b in batch], padding=True, return_tensors="pt")
        labels = lab.input_ids.masked_fill(lab.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.dsid).all():
            labels = labels[:, 1:]   # model menambahkan start token sendiri
        return {"input_features": feats, "labels": labels}

def _cleanup_training_gpu():
    for v in ["trainer", "proc_ft", "model_ft"]:
        if v in globals(): del globals()[v]
    gc.collect(); torch.cuda.empty_cache()

def run_finetune(model_id, ft_dir, drive_name, p_noise):
    _cleanup_training_gpu()
    proc = WhisperProcessor.from_pretrained(model_id, language=LANGUAGE, task="transcribe")
    mdl = WhisperForConditionalGeneration.from_pretrained(model_id)
    set_generation_config(mdl, GEN_KW_KEPT_BASE)
    print(f"{drive_name}: {sum(p.numel() for p in mdl.parameters())/1e6:.1f} juta parameter, p_noise={p_noise}")

    def compute_metrics(pred):
        ids = pred.label_ids.copy(); ids[ids == -100] = proc.tokenizer.pad_token_id
        P = [norm(t) for t in proc.batch_decode(pred.predictions, skip_special_tokens=True)]
        L = [norm(t) for t in proc.batch_decode(ids, skip_special_tokens=True)]
        keep = [i for i, l in enumerate(L) if l]
        return {"wer": jiwer.wer([L[i] for i in keep], [P[i] for i in keep])}

    args = Seq2SeqTrainingArguments(
        output_dir=ft_dir, per_device_train_batch_size=16, per_device_eval_batch_size=16,
        learning_rate=1e-5, warmup_steps=100, max_steps=MAX_STEPS, fp16=(DEVICE == "cuda"),
        eval_strategy="steps", eval_steps=500, save_steps=500, save_total_limit=2, logging_steps=25,
        predict_with_generate=True, generation_max_length=220,   # == max_new_tokens GEN_KW
        load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
        remove_unused_columns=False, dataloader_num_workers=2, report_to="none", seed=SEED)
    trainer = Seq2SeqTrainer(model=mdl, args=args, train_dataset=train_ds, eval_dataset=val_clean,
                            data_collator=Collator(proc, mdl.config.decoder_start_token_id,
                                                   p_noise=p_noise, bank=NOISE_BANK["train"]),
                            compute_metrics=compute_metrics)
    trainer.train()   # ~20-40 menit di T4 utk 1500 step (termasuk eval tiap 500)
    trainer.save_model(ft_dir); proc.save_pretrained(ft_dir)
    with open(f"{ft_dir}/log_history.json", "w") as f: json.dump(trainer.state.log_history, f)
    if os.path.isdir("/content/drive/MyDrive"):
        dst = f"{DRIVE_ROOT}/ckpt/{drive_name}"
        shutil.copytree(ft_dir, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
        print("Model + kurva training tersimpan di Drive:", dst)
    h = trainer.state.log_history
    tr = [(x["step"], x["loss"]) for x in h if "loss" in x]
    ev = [(x["step"], x["eval_wer"]) for x in h if "eval_wer" in x]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(*zip(*tr)); ax[0].set_title(f"Training loss ({drive_name})")
    if ev:
        ax[1].plot(*zip(*ev), marker="o"); ax[1].set_title(f"Val WER bersih ({drive_name}) — kurva utk laporan")
    plt.show()
    del trainer, mdl, proc
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ==== Model utama: whisper-base dengan augmentasi noise ====
FT_BASE = "/content/whisper-base-notulen-id"
run_finetune("openai/whisper-base", FT_BASE, "whisper-base-notulen-id", p_noise=P_NOISE)

In [ ]:
# ==== Ablation: whisper-base TANPA noise (untuk bukti (b)) ====
FT_ABL = "/content/whisper-base-notulen-id-cleanablation"
run_finetune("openai/whisper-base", FT_ABL, "whisper-base-notulen-id-cleanablation", p_noise=0.0)

In [ ]:
# ==== Cek ulang argumen decode utk model hasil fine-tune (masih hanya di validation) ====
FT_BASE = globals().get("FT_BASE", "/content/whisper-base-notulen-id")
proc_ft, mdl_ft = load_whisper(FT_BASE, GEN_KW_KEPT_BASE)
vc, vn, vr = build_val_audios()
w_ft = {}
for gname, gk in [("plain", GEN_KW_PLAIN), ("anti", GEN_KW_ANTI)]:
    m_c, _ = evaluate(proc_ft, mdl_ft, vc, vr, gk)
    m_n, _ = evaluate(proc_ft, mdl_ft, vn, vr, gk)
    w_ft[gname] = (m_c["wer"] + m_n["wer"]) / 2
    print(f"ft-base {gname}: val bersih {m_c['wer']:.4f} | babble10 {m_n['wer']:.4f} | rata2 {w_ft[gname]:.4f}")
win = min(w_ft, key=w_ft.get)
GEN_KW_BY_MODEL[FT_BASE] = GEN_KW_PLAIN if win == "plain" else GEN_KW_ANTI
if GEN_KW_BY_MODEL[FT_BASE] is not GEN_KW_KEPT_BASE:
    print(f"PERHATIAN: pemenang berubah jadi {win} — checkpoint dipilih dengan config lama; catat di laporan.")
else:
    print("Pemenang tetap sama dengan yang dipakai saat training.")
del mdl_ft; torch.cuda.empty_cache()

## 8. Evaluasi Gabungan (Tabel WER + CI, Kurva WER-vs-SNR, Contoh Hipotesis, Error Analysis)

Protokol:
- Semua skor memakai `norm()` dan **CI bootstrap 95%** (resample clip 1000x). Syarat (a) hanya gagal jika model fine-tuned lebih buruk **di luar** interval; hasil mepet dilaporkan apa adanya.
- Urutan loop: **test set -> kondisi (luar) -> model (dalam)**: set noise dibangun SEKALI per (test set, kondisi) lalu dievaluasi semua model, noise identik antar model; hanya satu set noise di memori.
- Noise uji hanya dari **bank test** (MUSAN 20% + OMEETING/OOFFICE/OHALLWAY + interferer hold-out) — tidak pernah dilihat training.
- Hasil ditulis ke CSV di Drive **di setiap kondisi** (aman dari reset), dan loop **bisa di-resume**: kombinasi yang sudah ada dilewati.
- 4 model dimuat fp16 sekaligus (~600 MB VRAM): pretrained-tiny, pretrained-base, ft-base, ablation-base-clean.

In [ ]:
# ==== Muat semua model sekali + loop evaluasi (resume-able) ====
for _g in ["TEST_SETS", "evaluate", "load_whisper", "NOISE_BANK", "CONDITIONS", "GEN_KW_PLAIN"]:
    assert _g in globals(), f"'{_g}' belum ada -> jalankan Bagian 1-4 dan cell harness Bagian 6 dulu."
GEN_KW_CACHE = globals().get("GEN_KW_CACHE", f"{DRIVE_ROOT}/logs/gen_kw_tuning.json")
GEN_KW_BY_MODEL = globals().get("GEN_KW_BY_MODEL", {})
if os.path.exists(GEN_KW_CACHE):
    for k, v in json.load(open(GEN_KW_CACHE)).items():
        GEN_KW_BY_MODEL.setdefault(k, GEN_KW_ANTI if v == "anti" else GEN_KW_PLAIN)
GEN_KW_KEPT_BASE = globals().get("GEN_KW_KEPT_BASE", GEN_KW_BY_MODEL.get("openai/whisper-base", GEN_KW_PLAIN))

def resolve_dir(local, drive_name):
    if os.path.exists(os.path.join(local, "config.json")): return local
    d = f"{DRIVE_ROOT}/ckpt/{drive_name}"
    assert os.path.exists(os.path.join(d, "config.json")), f"Model tidak ditemukan di {local} maupun {d}"
    return d

MODELS_EVAL = {
    "pretrained-tiny": ("openai/whisper-tiny", GEN_KW_BY_MODEL.get("openai/whisper-tiny", GEN_KW_PLAIN)),
    "pretrained-base": ("openai/whisper-base", GEN_KW_BY_MODEL.get("openai/whisper-base", GEN_KW_PLAIN)),
    "ft-base": (resolve_dir(globals().get("FT_BASE", "/content/whisper-base-notulen-id"), "whisper-base-notulen-id"),
                GEN_KW_BY_MODEL.get(globals().get("FT_BASE", "/content/whisper-base-notulen-id"), GEN_KW_KEPT_BASE)),
    "ablation-base-clean": (resolve_dir(globals().get("FT_ABL", "/content/whisper-base-notulen-id-cleanablation"), "whisper-base-notulen-id-cleanablation"),
                            GEN_KW_BY_MODEL.get(globals().get("FT_ABL", "/content/whisper-base-notulen-id-cleanablation"), GEN_KW_KEPT_BASE)),
}
loaded = {k: load_whisper(path, gk) for k, (path, gk) in MODELS_EVAL.items()}
print("Model termuat:", list(loaded))

EVAL_CSV = f"{DRIVE_ROOT}/eval/hasil_eval.csv"
rows, done = [], set()
if os.path.exists(EVAL_CSV):
    df_prev = pd.read_csv(EVAL_CSV)
    rows = df_prev.to_dict("records")
    done = set(zip(df_prev.test, df_prev.condition, df_prev.model))
    print(f"Resume: {len(done)} kombinasi sudah ada di CSV")

SAMPLES, FT_HYPS = [], {}
t_all = time.time()
for tname, tds in TEST_SETS.items():
    for cond in CONDITIONS:
        todo = [m for m in loaded if (tname, cond, m) not in done]
        if not todo: continue
        audios, refs = make_noisy_set(tds, cond, cond_seed(tname, cond), NOISE_BANK["test"])
        for mname in todo:
            proc, mdl = loaded[mname]
            m, (R, H, hyps) = evaluate(proc, mdl, audios, refs, MODELS_EVAL[mname][1])
            lo, hi = bootstrap_ci(m["counts"])
            rows.append(dict(test=tname, condition=cond, model=mname, wer=round(m["wer"], 4),
                             ci_lo=round(lo, 4), ci_hi=round(hi, 4), cer=round(m["cer"], 4), rtf=round(m["rtf"], 4)))
            if mname == "ft-base":
                if cond != "clean" and tname == "indspeech_holdout":
                    SAMPLES.append((tname, cond, list(zip(R[:5], H[:5]))))
                FT_HYPS[(tname, cond)] = (R, H)
            print(f"[{time.time()-t_all:7.0f}s] {tname:18s} {cond:10s} {mname:20s} "
                  f"WER={m['wer']:.3f} [{lo:.3f},{hi:.3f}] RTF={m['rtf']:.3f}")
        del audios, refs
        df = pd.DataFrame(rows)
        df.to_csv("/content/hasil_eval.csv", index=False)
        if os.path.isdir(DRIVE_ROOT): df.to_csv(EVAL_CSV, index=False)
print("Selesai. Total baris:", len(rows))

In [ ]:
# ==== Tabel + kurva WER-vs-SNR + contoh hipotesis ====
df = pd.read_csv(EVAL_CSV)
display(df.pivot_table(index=["test", "condition"], columns="model", values="wer").round(3))

for tname in df.test.unique():
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
    for i, kind in enumerate(NOISE_KINDS):
        for mname, g in df[df.test == tname].groupby("model"):
            pts = [(parse_cond(c)[1], w) for c, w in zip(g.condition, g.wer)
                   if parse_cond(c)[0] == kind]
            if pts: ax[i].plot(*zip(*sorted(pts)), marker="o", label=mname)
        ax[i].set_title(f"{tname} - {kind}"); ax[i].set_xlabel("SNR (dB)"); ax[i].set_ylabel("WER")
        ax[i].legend(fontsize=7)
    plt.tight_layout()
    if os.path.isdir(DRIVE_ROOT): plt.savefig(f"{DRIVE_ROOT}/eval/kurva_wer_{tname}.png", dpi=150)
    plt.show()

print("=== Contoh hipotesis (ft-base, indspeech_holdout, 5 per kondisi) ===")
for tname, cond, pairs in SAMPLES:
    print(f"--- {tname} | {cond} ---")
    for r, h in pairs:
        print(" REF:", r); print(" HYP:", h)
if os.path.isdir(DRIVE_ROOT):
    with open(f"{DRIVE_ROOT}/eval/contoh_hipotesis.txt", "w") as f:
        for tname, cond, pairs in SAMPLES:
            f.write(f"--- {tname} | {cond} ---\n")
            for r, h in pairs: f.write(f" REF: {r}\n HYP: {h}\n")

In [ ]:
# ==== Cek syarat (a) & (b) + error analysis ====
def status_vs(pre, ft):
    if ft.wer <= pre.wer: return "PASS"
    if ft.ci_lo <= pre.ci_hi: return "BORDERLINE (interval bertumpuk)"
    return "FAIL"

rows_a = []
for tname in df.test.unique():
    for cond in CONDITIONS:
        pre = df[(df.test == tname) & (df.condition == cond) & (df.model == "pretrained-base")].iloc[0]
        ft  = df[(df.test == tname) & (df.condition == cond) & (df.model == "ft-base")].iloc[0]
        rows_a.append(dict(test=tname, condition=cond, wer_pre=round(pre.wer, 4), wer_ft=round(ft.wer, 4),
                           delta=round(ft.wer - pre.wer, 4), status=status_vs(pre, ft)))
df_a = pd.DataFrame(rows_a)
display(df_a)
if os.path.isdir(DRIVE_ROOT): df_a.to_csv(f"{DRIVE_ROOT}/eval/cek_syarat_a.csv", index=False)

# Syarat (b): ft-base vs ablation di noise + biaya di bersih
abl = df[df.model == "ablation-base-clean"].set_index(["test", "condition"]).wer
ftb = df[df.model == "ft-base"].set_index(["test", "condition"]).wer
noisy = [c for c in CONDITIONS if c != "clean"]
print("Syarat (b): ft-base (noise-augmented) vs ablation (clean-only)")
for tname in df.test.unique():
    gain = float(np.mean([abl[(tname, c)] - ftb[(tname, c)] for c in noisy]))
    cost = float(ftb[(tname, "clean")] - abl[(tname, "clean")])
    print(f"  {tname:18s}: rata2 peningkatan di noise = +{gain:.4f} WER-ablation lebih buruk | biaya bersih = {cost:+.4f}")

# Error analysis (tabel v1): bersih + kondisi terburuk ft-base
def error_table(R, H, max_rows=25):
    rows_ = []
    for ref, hyp in zip(R, H):
        out = jiwer.process_words(ref, hyp)
        rw, hw = ref.split(), hyp.split()
        for ch in out.alignments[0]:
            if ch.type == "equal": continue
            r = " ".join(rw[ch.ref_start_idx:ch.ref_end_idx]); h = " ".join(hw[ch.hyp_start_idx:ch.hyp_end_idx])
            rows_.append(dict(reference=r or "-", output=h or "-",
                              error={"substitute": "Substitution", "delete": "Deletion", "insert": "Insertion"}[ch.type]))
    dfx = pd.DataFrame(rows_)
    print(dfx.error.value_counts().to_string())
    print("\nKata yang paling sering salah:")
    print(dfx[dfx.error != "Insertion"].reference.value_counts().head(15).to_string())
    return dfx.head(max_rows)

if FT_HYPS:
    ft_hold = df[(df.test == "indspeech_holdout") & (df.model == "ft-base")]
    worst = ft_hold.loc[ft_hold.wer.idxmax(), "condition"]
    print("\n=== Error analysis ft-base: indspeech_holdout BERSIH ===")
    R, H = FT_HYPS[("indspeech_holdout", "clean")]
    display(error_table(R, H))
    if (worst != "clean") and (("indspeech_holdout", worst) in FT_HYPS):
        print(f"\n=== Error analysis ft-base: indspeech_holdout KONDISI TERBURUK ({worst}) ===")
        R, H = FT_HYPS[("indspeech_holdout", worst)]
        display(error_table(R, H))

## 9. Simulasi Streaming (partial transcript, latency, RTF) — menyusul
Bagian ini akan ditambahkan pada tahap berikutnya (loop chunk 0.5 s, VAD state berjalan, partial tiap 1 s, finalisasi setelah hening 0.6 s / cap 20 s dengan potong di frame energi terendah, metrik latency + false activation). Replika dari algoritme yang akan dipakai web app.

## 10. Label Speaker ("siapa bilang apa") — menyusul
Embedding mean+std MFCC frame voiced dengan z-normalisasi statistik speaker kalibrasi, threshold pada EER kalibrasi (bukan tebakan 0.6), aturan online: centroid terdekat di atas threshold atau speaker baru. Evaluasi: meeting sintetis 5 speaker hold-out, akurasi segmen setelah pemetaan satu-ke-satu (Hungarian), jumlah speaker ditemukan vs sebenarnya.

## 11. Export ONNX (q8) + Upload Hugging Face Hub — menyusul
Per model: optimum `main_export` (Whisper: `automatic-speech-recognition-with-past` -> encoder+decoder_merged; Wav2Vec2-CTC: `automatic-speech-recognition` -> satu model.onnx) -> quantize uint8 -> layout Transformers.js -> verifikasi WER ONNX vs PyTorch (fp32 dan q8) -> Drive -> Hub (`{username}/whisper-tiny-notulen-id`, `whisper-base-notulen-id`, `wav2vec2-base-notulen-id`). Sebelum export: diagnosa dulu cacat v1 quantisasi decoder (Gather embedding table menyebabkan file tidak mengecil) dengan menghitung node terkuantisasi.

## Catatan: Resume Setelah Reset Runtime

Colab suka ter-reset (RAM habis / idle). Yang tersimpan di Drive: model (`ckpt/`), hasil eval (`eval/`), kurva training (`ckpt/*/log_history.json`), cache data (`data/`, `~8.5 GB`), cache tuning decode (`logs/`). Yang hilang: `/content` dan semua variabel Python.

| Situasi | Yang dijalankan ulang |
|---|---|
| Fresh run | Bagian 1-8 berurutan (pip -> import -> Drive -> data -> MFCC -> noise -> 13v12 -> baseline -> fine-tune -> evaluasi) |
| Reset saat/ setelah training | Bagian 1 (3 cell), Bagian 2 (data restore dari Drive), Bagian 3-4 (fungsi MFCC+noise), **cell harness Bagian 6** (norm/evaluate), lalu langsung Bagian 8 (model diambil otomatis dari Drive) |
| Reset saat evaluasi | sama seperti atas; loop evaluasi otomatis resume dari CSV Drive |
| Hanya mau export (nanti) | Bagian 1 saja + cell export Bagian 11 (menyusul) |

Hal yang tetap harus dikerjakan manual (di luar notebook): ukur latency di browser dari web app, cek kebijakan mata kuliah soal weight yang dilatih dengan data CC BY-NC-SA, cantumkan lisensi data di model card + laporan.